# 1. Data Processing

## Importing data

In [18]:
import os

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import lightgbm as lgb
import shap
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

In [19]:
DATA_PATH = "dataset/"

files = {}
for f in os.listdir(DATA_PATH):
    if f.endswith(".csv"):
        name = f.replace(".csv", "")
        files[name] = pd.read_csv(DATA_PATH + f)
        print(f"{f:35s} → {files[name].shape}")

web_traffic.csv                     → (3652, 7)
customers.csv                       → (121930, 7)
products.csv                        → (2412, 8)
reviews.csv                         → (113551, 7)
orders.csv                          → (646945, 8)
shipments.csv                       → (566067, 4)
test.csv                            → (548, 3)
promotions.csv                      → (50, 10)
geography.csv                       → (39948, 4)
payments.csv                        → (646945, 4)
order_items.csv                     → (714669, 7)
inventory.csv                       → (60247, 17)
sample_submission.csv               → (548, 3)
returns.csv                         → (39939, 7)
sales.csv                           → (3833, 3)


## Preprocessing

In [20]:
customers = files['customers']
geography = files['geography']
inventory = files['inventory']
orders = files['orders']
order_items = files['order_items']
payments = files['payments']
products = files['products']
promotions = files['promotions']
returns = files['returns']
reviews = files['reviews']
sales = files['sales']
shipments = files['shipments']
web_traffic = files['web_traffic']

In [21]:
# Convert date columns to datetime format

inventory['snapshot_date'] = pd.to_datetime(inventory['snapshot_date'])

orders['order_date'] = pd.to_datetime(orders['order_date'])

promotions['start_date'] = pd.to_datetime(promotions['start_date'])
promotions['end_date'] = pd.to_datetime(promotions['end_date'])

returns['return_date'] = pd.to_datetime(returns['return_date'])

reviews['review_date'] = pd.to_datetime(reviews['review_date'])

sales['Date'] = pd.to_datetime(sales['Date'])

shipments['ship_date'] = pd.to_datetime(shipments['ship_date'])
shipments['delivery_date'] = pd.to_datetime(shipments['delivery_date'])

web_traffic['date'] = pd.to_datetime(web_traffic['date'])

In [22]:
TEST_START = pd.to_datetime("2023-01-01")
TEST_END = pd.to_datetime("2024-07-01")

full_dates = pd.date_range(sales['Date'].min(), TEST_END, freq='D')
df = pd.DataFrame({'date': full_dates})

In [23]:
sales['gross_margin'] = (sales['Revenue'] - sales['COGS']) / sales['Revenue']

df = df.merge(sales, left_on='date', right_on='Date', how='left')
df.drop(columns='Date', inplace=True)

In [24]:
# Calendar features

df['day_of_week'] = df['date'].dt.dayofweek
df['day_sin'] = np.sin(2 * np.pi * df['date'].dt.day / 7)
df['day_cos'] = np.cos(2 * np.pi * df['date'].dt.day / 7)
df['days_since_start'] = (df['date'] - df['date'].min()).dt.days

df['month'] = df['date'].dt.month
df['month_sin'] = np.sin(2 * np.pi * df['date'].dt.month / 12)
df['month_cos'] = np.cos(2 * np.pi * df['date'].dt.month / 12)

df['is_weekend'] = df['day_of_week'].isin([4, 6]).astype(int)
df['post_2019'] = (df['date'] >= pd.to_datetime("2019-01-01")).astype(int)

# Lag features
for lag in [1, 2, 3, 7, 28, 30, 364, 728]:
    df[f'rev_lag{lag}'] = df['Revenue'].shift(lag)
    df[f'gm_lag{lag}'] = df['gross_margin'].shift(lag)

# Rolling features
df['rev_roll_med_7'] = df['Revenue'].shift(1).rolling(window=7).median()
df['rev_roll_med_28'] = df['Revenue'].shift(1).rolling(window=28).median()
df['rev_roll_ewma_7'] = df['Revenue'].shift(1).ewm(span=7, adjust=False).mean()
df['rev_roll_std_7'] = df['Revenue'].shift(1).rolling(window=7).std()
df['rev_roll_std_28'] = df['Revenue'].shift(1).rolling(window=28).std()

df['gm_roll_med_7'] = df['gross_margin'].shift(1).rolling(window=7).median()
df['gm_roll_med_28'] = df['gross_margin'].shift(1).rolling(window=28).median()
df['gm_roll_ewma_7'] = df['gross_margin'].shift(1).ewm(span=7, adjust=False).mean()
df['gm_roll_std_7'] = df['gross_margin'].shift(1).rolling(window=7).std()
df['gm_roll_std_28'] = df['gross_margin'].shift(1).rolling(window=28).std()
df['gm_roll_min_7'] = df['gross_margin'].shift(1).rolling(window=7).min()

df['roll_corr_rev_gm_7'] = df['Revenue'].shift(1).rolling(window=7).corr(df['gross_margin']) #Subject to be replaced by is promo active in the future


In [25]:
# Fill NaN
fill_cols = [c for c in df.columns if c not in ['Date','Revenue','COGS','GrossMargin']]
df[fill_cols] = df[fill_cols].fillna(method='ffill').fillna(method='bfill')

In [26]:
BASE_CALENDAR_FEATURES = ['day_of_week', 'day_sin', 'day_cos', 'days_since_start', 'month', 'month_sin', 'month_cos', 'is_weekend', 'post_2019']
LAG_FEATURES = [f'rev_lag{lag}' for lag in [1, 2, 3, 7, 28, 30, 364, 728]] + [f'gm_lag{lag}' for lag in [1, 2, 3, 7, 28, 30, 364, 728]]
ROLLING_FEATURES = ['rev_roll_med_7', 'rev_roll_med_28', 'rev_roll_ewma_7', 'rev_roll_std_7', 'rev_roll_std_28', 'gm_roll_med_7', 'gm_roll_med_28', 'gm_roll_ewma_7', 'gm_roll_std_7', 'gm_roll_std_28', 'gm_roll_min_7', 'roll_corr_rev_gm_7']

features = BASE_CALENDAR_FEATURES + LAG_FEATURES + ROLLING_FEATURES

In [28]:
train_df = df[df['date'] < TEST_START].copy()
test_df = df[(df['date'] >= TEST_START) & (df['date'] <= TEST_END)].copy()

X_train_rev = train_df[features]
X_train_gm = train_df[features]

y_train_rev = train_df['Revenue']
y_train_gm = train_df['gross_margin']


# LightGBM training

## Metrics

In [29]:
def evaluate(y_true, y_pred, label=''):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"  {label:30s} MAE={mae:>12,.2f}  RMSE={rmse:>12,.2f}  R²={r2:.4f}")
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

In [ ]:
LGBM_PARAMS = dict(
    n_estimators    = 2000,
    learning_rate   = 0.01,
    num_leaves      = 31,
    min_child_samples = 20,
    subsample       = 0.8,
    colsample_bytree = 0.8,
    reg_alpha       = 0.1,
    reg_lambda      = 0.1,
    random_state    = 42,
    n_jobs          = -1,
    verbose         = -1
)

tscv = TimeSeriesSplit(n_splits=5)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_rev)):
    X_tr_rev, X_val_rev = X_train_rev.iloc[train_idx], X_train_rev.iloc[val_idx]
    y_tr_rev, y_val_rev = y_train_rev.iloc[train_idx], y_train_rev.iloc[val_idx]
    
    model_rev = lgb.LGBMRegressor(**LGBM_PARAMS)
    model_rev.fit(X_tr_rev, y_tr_rev, eval_set=[(X_val_rev, y_val_rev)], early_stopping_rounds=100, verbose=False)
    
    val_pred = model_rev.predict(X_val_rev)
    scores = evaluate(y_val_rev, val_pred, label=f'Fold {fold+1}')
    cv_scores.append(scores)